<h2>DDPM</h2>

* Computed dataset mean and std from training set only
* Applied normalization using these statistics
* Added data augmentations (horizontal flip, color jitter) to training set
2. **DDPM Implementation**
* Built U-Net with:
  * Sinusoidal time embeddings
  * Residual blocks and skip connections
  * Attention blocks at selected scales
* Implemented forward diffusion (`q_sample`) and reverse sampling (`p_sample`)
* Supported both linear and cosine beta schedules
* Used mixed precision (AMP) to accelerate training
3. **Training Setup**
* Trained on 64x64 resolution images
* Used:
  * AdamW optimizer with weight decay
  * Cosine annealing learning rate schedule
  * Gradient clipping
* Saved intermediate checkpoints and samples during training
4. **Evaluation**
  * FID (Fréchet Inception Distance)
  * Inception Score (IS)
* Generated sample images (up to 500) for evaluation
 5. **Key Results**
  * FID: 303.69
  * Inception Score: 2.47 ± 0.23
  * Sample quality: recognizable basic shapes/colors, limited detail

<h4>Model Tuning</h4>
Steps Implemented:

* Enhanced data augmentation RandomCrop / RandomResizedCrop → improves robustness to scale & position
* ColorJitter → better color diversity
* RandomRotation → helps orientation robustness
* GaussianBlur / RandomAffine → mild distortions to prevent overfitting
* Increase UNet depth (more conv layers, residual blocks, attention at 16x16 or 32x32) Try cosine beta schedule instead of linear Use EMA (Exponential Moving Average) of model weights for better sampling.
* Train longer (more epochs) with dynamic LR scheduling (ReduceLROnPlateau).

In [1]:
import torch
from torch import nn
from torchvision import transforms, utils
import matplotlib.pyplot as plt
import os
from torchvision.datasets import ImageFolder
import torch.optim as optim
import torch.nn.functional as F
import time
from cleanfid import fid
from tqdm import tqdm
import numpy as np
from torch.utils.data import random_split
from PIL import Image
import torchvision
from torchvision.models import inception_v3
import os
import torch
import numpy as np
from torchvision.utils import save_image

In [2]:

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Create directories for evaluation
os.makedirs("generated_images", exist_ok=True)
os.makedirs("real_images_test", exist_ok=True)

Using device: cuda


In [3]:
# Load datasets without data leakage
train_dataset = ImageFolder(root='sampled/64_pix/split_photos_64/train')  
test_dataset = ImageFolder(root='sampled/64_pix/split_photos_64/test')  

# Compute mean and std from training set only (avoid data leakage)
print("Computing dataset statistics from training set...")
train_transform = transforms.Compose([transforms.Resize((64, 64)), transforms.ToTensor()])
train_dataset.transform = train_transform


Computing dataset statistics from training set...


In [4]:
# Calculate mean and std
loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, num_workers=4)
mean = 0.
std = 0.
nb_samples = 0.
for data, _ in tqdm(loader):
    batch_samples = data.size(0)
    data = data.view(batch_samples, data.size(1), -1)
    mean += data.mean(2).sum(0)
    std += data.std(2).sum(0)
    nb_samples += batch_samples

mean /= nb_samples
std /= nb_samples
print(f"Training set - Mean: {mean}, Std: {std}")

100%|██████████████████████████████████████████████████████████████████████████████| 1112/1112 [02:51<00:00,  6.48it/s]

Training set - Mean: tensor([0.4986, 0.4289, 0.3752]), Std: tensor([0.2265, 0.2231, 0.2285])


In [5]:
# Apply normalization using training set statistics
normalize = transforms.Normalize(mean.tolist(), std.tolist())
denormalize = transforms.Normalize(
    mean=[-m/s for m, s in zip(mean, std)],
    std=[1/s for s in std]
)

# Final transforms
train_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    normalize
])

test_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    normalize
])

# Apply transforms
train_dataset.transform = train_transform
test_dataset.transform = test_transform

In [6]:

# Create dataloaders
dataloader = torch.utils.data.DataLoader(
    train_dataset, batch_size=64, 
    shuffle=True, num_workers=4, pin_memory=True
)

In [7]:
# Save test images for evaluation
print("Saving test images for evaluation...")
for idx, (img, _) in enumerate(test_dataset):
    # Denormalize and save
    denorm_img = denormalize(img).clamp(0, 1)
    utils.save_image(denorm_img, f"real_images_test/img_{idx:05d}.png")


Saving test images for evaluation...


In [8]:
# Diffusion parameters
T = 1000  # number of diffusion steps

def linear_beta_schedule(timesteps):
    beta_start = 0.0001
    beta_end = 0.02
    return torch.linspace(beta_start, beta_end, timesteps)

# Precompute all diffusion parameters
betas = linear_beta_schedule(T).to(device)
alphas = 1. - betas
alphas_cumprod = torch.cumprod(alphas, axis=0)
alphas_cumprod_prev = torch.cat([torch.tensor([1.]).to(device), alphas_cumprod[:-1]])
sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod)
sqrt_one_minus_alphas_cumprod = torch.sqrt(1 - alphas_cumprod)
recip_sqrt_alphas = 1.0 / torch.sqrt(alphas)

In [9]:
# Time embedding
class SinusoidalPosEmb(nn.Module):
    def __init__(self, dim=16):
        super().__init__()
        self.dim = dim

    def forward(self, x):
        device = x.device
        half_dim = self.dim // 2
        emb = torch.log(torch.tensor(10000.)) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=device) * -emb)
        emb = x[:, None] * emb[None, :]
        return torch.cat([emb.sin(), emb.cos()], dim=-1)

In [10]:
class SimpleUNet(nn.Module):
    def __init__(self, in_channels=3, time_dim=32):
        super().__init__()
        
        # Time embedding layers
        self.time_mlp = nn.Sequential(
            SinusoidalPosEmb(time_dim),
            nn.Linear(time_dim, time_dim * 4),
            nn.ReLU(),
            nn.Linear(time_dim * 4, time_dim)
        )
        
        # First half (downsampling path)
        self.conv1 = nn.Conv2d(in_channels, 64, 3, padding=1)
        self.conv2 = nn.Conv2d(64, 128, 3, padding=1)
        self.conv3 = nn.Conv2d(128, 256, 3, padding=1)
        self.down_pool = nn.MaxPool2d(2)
        
        # Bottleneck at 16x16 resolution
        self.bottleneck = nn.Conv2d(256, 256, 3, padding=1)
        
        # Second half (upsampling path)
        self.up_conv1 = nn.ConvTranspose2d(256, 128, 2, stride=2)  # Upsample to 32x32
        self.conv4 = nn.Conv2d(256, 128, 3, padding=1)  # 128 from up + 128 from skip
        
        self.up_conv2 = nn.ConvTranspose2d(128, 64, 2, stride=2)  # Upsample to 64x64
        self.conv5 = nn.Conv2d(128, 64, 3, padding=1)   # 64 from up + 64 from skip
        
        self.conv6 = nn.Conv2d(64, in_channels, 3, padding=1)
        
        # Time projection layers
        self.time_proj1 = nn.Linear(time_dim, 64)
        self.time_proj2 = nn.Linear(time_dim, 128)
        self.time_proj3 = nn.Linear(time_dim, 256)

    def forward(self, x, t):
        t_emb = self.time_mlp(t)
        
        # Downsample path
        x1 = F.relu(self.conv1(x))
        x1 = x1 + self.time_proj1(t_emb)[:, :, None, None]
        
        x2 = self.down_pool(x1)  # 64x64 -> 32x32
        x2 = F.relu(self.conv2(x2))
        x2 = x2 + self.time_proj2(t_emb)[:, :, None, None]
        
        x3 = self.down_pool(x2)  # 32x32 -> 16x16
        x3 = F.relu(self.conv3(x3))
        x3 = x3 + self.time_proj3(t_emb)[:, :, None, None]
        
        # Bottleneck at 16x16
        x4 = F.relu(self.bottleneck(x3))
        
        # Upsample path with skip connections
        # Step 1: Upsample from 16x16 to 32x32
        x5 = self.up_conv1(x4)
        # Crop and concatenate with x2 (skip connection)
        x5 = torch.cat([x5, x2], dim=1)
        x5 = F.relu(self.conv4(x5))
        
        # Step 2: Upsample from 32x32 to 64x64
        x6 = self.up_conv2(x5)
        # Crop and concatenate with x1 (skip connection)
        x6 = torch.cat([x6, x1], dim=1)
        x6 = F.relu(self.conv5(x6))
        
        # Final convolution
        x7 = self.conv6(x6)
        
        return x7

In [11]:

# Initialize model
model = SimpleUNet().to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-4)
scheduler = optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.98)
print(f"Model parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")


Model parameters: 1.52M


In [12]:
# Diffusion functions
def q_sample(x_start, t, noise=None):
    if noise is None:
        noise = torch.randn_like(x_start)
    
    sqrt_alpha_hat = sqrt_alphas_cumprod[t][:, None, None, None]
    sqrt_one_minus_alpha_hat = sqrt_one_minus_alphas_cumprod[t][:, None, None, None]
    return sqrt_alpha_hat * x_start + sqrt_one_minus_alpha_hat * noise

@torch.no_grad()
def p_sample(model, x, t):
    betas_t = betas[t][:, None, None, None]
    sqrt_one_minus_alpha_hat_t = sqrt_one_minus_alphas_cumprod[t][:, None, None, None]
    recip_sqrt_alpha_t = recip_sqrt_alphas[t][:, None, None, None]
    
    model_output = model(x, t)
    pred_noise = model_output
    
    model_mean = recip_sqrt_alpha_t * (x - betas_t * pred_noise / sqrt_one_minus_alpha_hat_t)
    
    if t[0] == 0:
        return model_mean
    else:
        posterior_variance = betas_t
        noise = torch.randn_like(x)
        return model_mean + torch.sqrt(posterior_variance) * noise
@torch.no_grad()
def sample_ddpm(model, num_samples=16):
    model.eval()
    shape = (num_samples, 3, 64, 64)
    img = torch.randn(shape).to(device)
    
    for i in tqdm(reversed(range(T)), desc="Sampling", total=T):
        t = torch.full((num_samples,), i, dtype=torch.long).to(device)
        img = p_sample(model, img, t)
    
    # Denormalize and clamp to [0,1]
    img = denormalize(img).clamp(0, 1)
    return img


In [13]:
# Inception Score implementation
def compute_inception_score(images, batch_size=32, resize=True, splits=10):
    """Compute Inception Score for generated images"""
    # Load pretrained Inception model
    inception_model = inception_v3(pretrained=True, transform_input=False).to(device)
    inception_model.eval()
    
    # Preprocessing function
    preprocess = transforms.Compose([
        transforms.Resize((299, 299)) if resize else transforms.Lambda(lambda x: x),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    # Get predictions
    preds = []
    n_batches = int(np.ceil(len(images) / batch_size))
    
    for i in range(n_batches):
        start_idx = i * batch_size
        end_idx = min((i + 1) * batch_size, len(images))
        batch = images[start_idx:end_idx].to(device)
        batch = preprocess(batch)
        
        with torch.no_grad():
            pred = inception_model(batch)
            pred = F.softmax(pred, dim=1).cpu().numpy()
            preds.append(pred)
    
    preds = np.concatenate(preds, axis=0)
    
    # Compute scores
    scores = []
    for i in range(splits):
        part = preds[i * len(preds) // splits: (i + 1) * len(preds) // splits]
        kl = part * (np.log(part) - np.log(np.expand_dims(np.mean(part, 0), 0)))
        kl = np.mean(np.sum(kl, 1))
        scores.append(np.exp(kl))
    
    return np.mean(scores), np.std(scores)

In [14]:
# Evaluation functions
def compute_fid():
    fid_score = fid.compute_fid(
        "generated_images", 
        "real_images_test",
        device=device,
        num_workers=0,
        verbose=False
    )
    print(f"FID Score: {fid_score:.2f}")
    return fid_score

def compute_is(num_samples=500, batch_size=32):
    # Generate samples
    all_samples = []
    n_batches = int(np.ceil(num_samples / batch_size))
    
    for i in range(n_batches):
        samples = sample_ddpm(model, num_samples=min(batch_size, num_samples - i * batch_size))
        all_samples.append(samples)
    
    # Combine and compute IS
    all_samples = torch.cat(all_samples, dim=0)
    is_mean, is_std = compute_inception_score(all_samples, batch_size=batch_size)
    print(f"Inception Score: {is_mean:.2f} ± {is_std:.2f}")
    return is_mean


In [17]:
def train_ddpm(model, dataloader, optimizer, epochs, time_limit=None):
    model.train()
    start_time = time.time()
    step = 0

    for epoch in range(epochs):
        pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{epochs}")
        epoch_loss = 0
        for images, _ in pbar:
            step += 1
            images = images.to(device)
            t = torch.randint(0, T, (images.size(0),)).long().to(device)

            noise = torch.randn_like(images)
            noisy_images = q_sample(images, t, noise)
            pred_noise = model(noisy_images, t)
            loss = F.mse_loss(pred_noise, noise)
            epoch_loss += loss.item()

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            pbar.set_postfix(loss=loss.item())

            if time_limit is not None:
                elapsed = time.time() - start_time
                if elapsed > time_limit:
                    print(f"\n⏰ Time limit reached at {elapsed/60:.1f} minutes")
                    return

        torch.save(model.state_dict(), f"ddpm_epoch_{epoch}.pth")

        if epoch % 5 == 0:
            samples = sample_ddpm(model, num_samples=16)
            utils.save_image(samples, f"generated_epoch_{epoch}.png", nrow=4)
            for i in range(samples.size(0)):
                utils.save_image(
                    samples[i], 
                    f"generated_images/epoch_{epoch}_sample_{i}.png"
                )

        scheduler.step()
        elapsed = time.time() - start_time
        avg_loss = epoch_loss / len(dataloader)
        print(f"Epoch {epoch+1} | Loss: {avg_loss:.4f} | Time: {elapsed/60:.1f} min")

    print("✅ Training completed")


In [47]:
# Run training
if __name__ == "__main__":
    # Clear generated images directory
    for f in os.listdir("generated_images"):
        os.remove(os.path.join("generated_images", f))
    
    # Train for 2 hours
    train_ddpm(
        model=model,
        dataloader=dataloader,
        optimizer=optimizer,
        epochs=100,
        time_limit=2*60*60  # 2 hours
    )
    
    # Generate 5 final images
    print("Generating 5 final samples...")
    samples = sample_ddpm(model, num_samples=5)
    
    # Save the 5 generated images
    for i in range(5):
        utils.save_image(
            samples[i], 
            f"generated_images/final_sample_{i:04d}.png"
        )
    
    # Save a grid of all 5 images
    utils.save_image(samples, "final_samples_5.png", nrow=5)
    
    print("\n5 Generated Images Saved:")
    print("Individual images: generated_images/final_sample_0000.png to final_sample_0004.png")
    print("Grid image: final_samples_5.png")

        # Compute metrics
    print("\nComputing evaluation metrics...")
    fid_score = compute_fid()
    is_score = compute_is(num_samples=500)
    
    print("\nFinal Results:")
    print(f"FID Score: {fid_score:.2f}")
    print(f"Inception Score: {is_score:.2f}")

Sampling: 100%|████████████████████████████████████████████████████████████████████| 1000/1000 [00:20<00:00, 48.87it/s]


Epoch 1 | Loss: 0.1938 | Time: 4.4 min


Epoch 2/100: 100%|█████████████████████████████████████████████████████| 1112/1112 [03:58<00:00,  4.65it/s, loss=0.092]


Epoch 2 | Loss: 0.0983 | Time: 8.3 min


Epoch 3/100: 100%|████████████████████████████████████████████████████| 1112/1112 [04:01<00:00,  4.61it/s, loss=0.0601]


...

Epoch 30/100:  49%|██████████████████████████▏                          | 549/1112 [02:14<02:18,  4.08it/s, loss=0.051]


Generating 5 final samples...


Sampling: 100%|███████████████████████████████████████████████████████████████████| 1000/1000 [00:09<00:00, 108.36it/s]



5 Generated Images Saved:
Individual images: generated_images/final_sample_0000.png to final_sample_0004.png
Grid image: final_samples_5.png

Computing evaluation metrics...


AttributeError: Can't pickle local object 'make_resizer.<locals>.func'

In [49]:
# === Evaluation Block (Run Separately) ===

print("\n📊 Computing Evaluation Metrics...")

# FID Score
fid_score = compute_fid()

# Inception Score
is_score = compute_is(num_samples=500)

# Final Results
print(f"""
🏁 Final Results:
-------------------------------
FID Score:        {fid_score:.2f}
Inception Score:  {is_score:.2f}
-------------------------------
""")



📊 Computing Evaluation Metrics...
FID Score: 303.69


Sampling: 100%|████████████████████████████████████████████████████████████████████| 1000/1000 [00:31<00:00, 31.27it/s]
C:\Users\safna\anaconda3\envs\py310\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\safna\anaconda3\envs\py310\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=Inception_V3_Weights.IMAGENET1K_V1`. You can also use `weights=Inception_V3_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/inception_v3_google-0cc3c7bd.pth" to C:\Users\safna/.cache\torch\hub\checkpoints\inception_v3_google-0cc3c7bd.pth
100%|████████████████████████████████████████████████████

Inception Score: 2.47 ± 0.23

🏁 Final Results:
-------------------------------
FID Score:        303.69
Inception Score:  2.47
-------------------------------



DDPM model is producing recognizable but low-fidelity images.

* The high FID and low IS suggest the model is in an early or undertrained state.
* DDPMs are capable of high performance but need:
* More diffusion steps or longer training time
* Improved denoising networks (e.g., UNet variants)
* Better label conditioning or noise schedule tuning